# Bloque 0 – Dependencias, rutas y parámetros fiscales

In [20]:
# libraries
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# -----------------------------------------------------
# PATH CONFIGURATION
# -----------------------------------------------------
ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

from config.paths import RAW_DIR, PROCESSED_DIR

# -----------------------------------------------------
# TAX BRACKETS BY YEAR
# -----------------------------------------------------
TAX_BRACKETS = {
    2018: [
        {"li": 0.01, "ls": 8952.49, "cf": 0, "tm": 0.0192},
        {"li": 8952.50, "ls": 75984.55, "cf": 171.88, "tm": 0.064},
        {"li": 75984.56, "ls": 133536.07, "cf": 4461.94, "tm": 0.1088},
        {"li": 133536.08, "ls": 155229.80, "cf": 10723.55, "tm": 0.16},
        {"li": 155229.81, "ls": 185852.57, "cf": 14194.54, "tm": 0.1792},
        {"li": 185852.58, "ls": 374837.88, "cf": 19682.13, "tm": 0.2136},
        {"li": 374837.89, "ls": 590795.99, "cf": 60049.40, "tm": 0.2352},
        {"li": 590796.00, "ls": 1127926.84, "cf": 110842.74, "tm": 0.30},
        {"li": 1127926.85, "ls": 1503902.46, "cf": 271981.99, "tm": 0.32},
        {"li": 1503902.47, "ls": 4511707.37, "cf": 392294.17, "tm": 0.34},
        {"li": 4511707.38, "ls": np.inf, "cf": 1414947.85, "tm": 0.35},
    ],
    2024: [
        {"li": 0.01, "ls": 8952.49, "cf": 0, "tm": 0.0192},
        {"li": 8952.50, "ls": 75984.55, "cf": 171.88, "tm": 0.064},
        {"li": 75984.56, "ls": 133536.07, "cf": 4461.94, "tm": 0.1088},
        {"li": 133536.08, "ls": 155229.80, "cf": 10723.55, "tm": 0.16},
        {"li": 155229.81, "ls": 185852.57, "cf": 14194.54, "tm": 0.1792},
        {"li": 185852.58, "ls": 374837.88, "cf": 19682.13, "tm": 0.2136},
        {"li": 374837.89, "ls": 590795.99, "cf": 60049.40, "tm": 0.2352},
        {"li": 590796.00, "ls": 1127926.84, "cf": 110842.74, "tm": 0.30},
        {"li": 1127926.85, "ls": 1503902.46, "cf": 271981.99, "tm": 0.32},
        {"li": 1503902.47, "ls": 4511707.37, "cf": 392294.17, "tm": 0.34},
        {"li": 4511707.38, "ls": np.inf, "cf": 1414947.85, "tm": 0.35},
    ]
}

# -----------------------------------------------------
# DEFLATORS (base 2018)
# -----------------------------------------------------
DEFLATORS = {
    2018: 1.0,
    2024: (100.0 / 133.543876114864)
}

# Bloque 1 – Clasificación de claves ENIGH por componente CEQ

In [21]:
# =====================================================
# INCOME COMPONENT MAPPINGS (by year)
# =====================================================

# Labor income keys (net amounts reported in ing_tri)
LABOR_KEYS = [
    "P001", "P002", "P003", "P004", "P005", "P006", "P007",
    "P008", "P009", "P011", "P012", "P013",  # main job
    "P014", "P015", "P016", "P018", "P019", "P020", "P021", "P022",  # secondary job
    "P067",  # income of children under 12
    "P068", "P069", "P070", "P071", "P072", "P073", "P074", "P075",
    "P076", "P077", "P078", "P079", "P080", "P081"  # self-employment
]
# Note: P020 disappears in 2024, but we can keep it in list; it just won't appear.

# Capital income keys
CAPITAL_KEYS = [
    "P023", "P024", "P025", "P026", "P027", "P028", "P029", "P030", "P031",
    "P050"  # P050 marked as DUDA, included for completeness
]

# Private transfers keys
PRIVATE_TRANSFER_KEYS = ["P039", "P040", "P041"]

# Public direct transfers – 2018
DIRECT_TRANSFERS_2018 = [
    "P042",  # Prospera
    "P043",  # PROCAMPO
    "P044",  # Pensión Adultos Mayores
    "P045",  # Otros programas para adultos mayores
    "P046",  # Tarjeta SinHambre
    "P047",  # Empleo temporal
    "P048",  # Otros programas sociales
    "P038"   # Becas de gobierno (no Prospera)
]

# Public direct transfers – 2024
DIRECT_TRANSFERS_2024 = [
    "P101",  # Becas Benito Juárez (básica)
    "P102",  # Beca Universal Educación Media Superior
    "P103",  # Jóvenes Escribiendo el Futuro
    "P104",  # Pensión Adultos Mayores (65 y más)
    "P105",  # Pensión para personas con discapacidad
    "P106",  # Apoyo estancias infantiles / madres trabajadoras
    "P107",  # Seguro de Vida para Jefas de Familia
    "P108",  # Jóvenes Construyendo el Futuro
    "P043",  # PROCAMPO / Producción para el Bienestar
    "P045",  # Otros programas para adultos mayores
    "P038"   # Becas de gobierno (no Prospera)
]

# Excluded keys (not used in income definition)
EXCLUDED_KEYS = [
    "P049", "P051", "P052", "P053", "P054", "P055", "P056", "P057",
    "P058", "P059", "P060", "P061", "P062", "P063", "P064", "P065", "P066"
]

# Bloque 2 – Funciones de cálculo (gross‑up laboral y cálculo de ISR)

In [22]:
# =====================================================
# FUNCTIONS
# =====================================================

def gross_up_labor(net_series, brackets):
    """
    Given a Series of net labour income,
    returns a Series with gross labour income
    using the inverted tax bracket formula.
    Handles zero and NaN by setting gross to 0.
    """
    gross = pd.Series(np.nan, index=net_series.index)
    
    # Fill missing net with 0 temporarily
    net = net_series.fillna(0)
    
    for b in brackets:
        gross_temp = (
            net + b["cf"] - b["tm"] * b["li"]
        ) / (1 - b["tm"])
        
        mask = (
            gross.isna()
            & (gross_temp >= b["li"])
            & (gross_temp <= b["ls"])
        )
        gross[mask] = gross_temp[mask]
    
    # For net == 0, the formula gives a small negative value -> not matched, so remain NaN.
    # Set them explicitly to 0.
    gross[net == 0] = 0.0
    
    # Any remaining NaN (should not happen) set to 0 as safety
    gross = gross.fillna(0)
    
    return gross


def calculate_isr(total_income, brackets):
    """
    Compute ISR for a given total income using the tax schedule.
    Returns the tax amount.
    """
    if total_income <= 0:
        return 0.0
    for b in brackets:
        if b["li"] <= total_income <= b["ls"]:
            return b["cf"] + b["tm"] * (total_income - b["li"])
    # If no bracket matched (should not happen), return 0
    return 0.0

# Bloque 3 – Definición de estructuras y tipos (para futuro)

In [23]:
# =====================================================
# BLOQUE 2
# DEFINIR ESTRUCTURA Y TIPOS
# =====================================================

# -----------------------------
# IDs
# -----------------------------
ID_TYPES = {
    "folioviv": "string",
    "foliohog": "string",
    "numren": "int64"
}

# -----------------------------
# POBLACION
# -----------------------------
POBLACION_COLUMNS = [
    "folioviv",
    "foliohog",
    "numren",
    "sexo",
    "edad",
    "parentesco"
]

POBLACION_TYPES = {
    "sexo": "int64",
    "edad": "int64",
    "parentesco": "int64"
}

# -----------------------------
# CONCENTRADO HOGAR
# -----------------------------
HOGAR_COLUMNS = [
    "folioviv",
    "foliohog",
    "ubica_geo",
    "est_dis",
    "upm",
    "factor"
]

HOGAR_TYPES = {
    "ubica_geo": "int64",
    "est_dis": "int64",
    "upm": "int64",
    "factor": "float64"
}

# -----------------------------
# INCOME (final columns)
# -----------------------------
INCOME_COLUMNS = [
    "folioviv",
    "foliohog",
    "numren",
    "mi_capital",
    "mi_private_transfers",
    "market_income",
    "net_market_income",
    "di_direct_transfers",
    "disposable_income"
]

INCOME_TYPES = {
    "mi_capital": "float64",
    "mi_private_transfers": "float64",
    "market_income": "float64",
    "net_market_income": "float64",
    "di_direct_transfers": "float64",
    "disposable_income": "float64"
}

print("Structure defined successfully")

Structure defined successfully


# Bloque 4 – Procesamiento ENIGH 2018

In [24]:
# =====================================================
# BLOQUE 4
# PROCESAMIENTO ENIGH 2018
# =====================================================

year = 2018
print(f"\n{'='*30}")
print(f"Processing {year}")
print(f"{'='*30}")

# ---- Load data ----
df = pd.read_csv(
    RAW_DIR / f"ingresos_{year}.csv",
    dtype={
        "folioviv": "string",
        "foliohog": "string",
        "numren": "int64",
        "clave": "string",
        "ing_tri": "float64"
    },
    low_memory=False
)
print(f"Rows raw: {len(df)}")

# ---- Classify each record ----
# Labor
mask_labor = df["clave"].isin(LABOR_KEYS)
# Capital
mask_capital = df["clave"].isin(CAPITAL_KEYS)
# Private transfers
mask_private = df["clave"].isin(PRIVATE_TRANSFER_KEYS)
# Direct transfers (2018 list)
mask_direct = df["clave"].isin(DIRECT_TRANSFERS_2018)
# (Excluded keys are ignored)

# ---- Aggregate per person ----
# Net labor income (sum of ing_tri for labor records)
net_labor = df.loc[mask_labor].groupby(
    ["folioviv", "foliohog", "numren"]
)["ing_tri"].sum().reset_index(name="net_labor")

# Capital
mi_capital = df.loc[mask_capital].groupby(
    ["folioviv", "foliohog", "numren"]
)["ing_tri"].sum().reset_index(name="mi_capital")

# Private transfers
mi_private = df.loc[mask_private].groupby(
    ["folioviv", "foliohog", "numren"]
)["ing_tri"].sum().reset_index(name="mi_private_transfers")

# Direct transfers
di_direct = df.loc[mask_direct].groupby(
    ["folioviv", "foliohog", "numren"]
)["ing_tri"].sum().reset_index(name="di_direct_transfers")

# ---- Merge all components ----
person_base = df[["folioviv", "foliohog", "numren"]].drop_duplicates()
person = person_base.merge(net_labor, on=["folioviv","foliohog","numren"], how="left")
person = person.merge(mi_capital, on=["folioviv","foliohog","numren"], how="left")
person = person.merge(mi_private, on=["folioviv","foliohog","numren"], how="left")
person = person.merge(di_direct, on=["folioviv","foliohog","numren"], how="left")

# Fill NaN with 0
person["net_labor"] = person["net_labor"].fillna(0)
person["mi_capital"] = person["mi_capital"].fillna(0)
person["mi_private_transfers"] = person["mi_private_transfers"].fillna(0)
person["di_direct_transfers"] = person["di_direct_transfers"].fillna(0)

# ---- Gross-up labor ----
brackets = TAX_BRACKETS[year]
person["mi_trabajo"] = gross_up_labor(person["net_labor"], brackets)

# ---- Market income (total) ----
person["market_income"] = (
    person["mi_trabajo"]
    + person["mi_capital"]
    + person["mi_private_transfers"]
)

# ---- ISR on total market income ----
person["isr"] = person["market_income"].apply(lambda x: calculate_isr(x, brackets))

# ---- Net market income ----
person["net_market_income"] = person["market_income"] - person["isr"]

# ---- Disposable income ----
person["disposable_income"] = person["net_market_income"] + person["di_direct_transfers"]

# ---- Deflate to 2018 prices ----
deflator = DEFLATORS[year]
monetary_cols = ["mi_trabajo", "mi_capital", "mi_private_transfers",
                 "market_income", "net_market_income",
                 "di_direct_transfers", "disposable_income"]
person[monetary_cols] = person[monetary_cols] * deflator

# ---- Keep final columns (as requested) ----
# The user requested: folioviv|foliohog|numren|year|mi_capital|mi_private_transfers|
# market_income|net_market_income|di_direct_transfers|disposable_income
person["year"] = year
df_2018_out = person[["folioviv", "foliohog", "numren", "year", "mi_trabajo", "isr",
                      "mi_capital", "mi_private_transfers",
                      "market_income", "net_market_income",
                      "di_direct_transfers", "disposable_income"]].copy()

# ---- Diagnostics ----
print(f"Persons: {len(df_2018_out)}")
print(df_2018_out[["market_income", "net_market_income", "disposable_income"]].describe())

# ---- Save ----
output_file = PROCESSED_DIR / f"income_components_{year}.csv"
df_2018_out.to_csv(output_file, index=False)
print(f"Saved: {output_file.name}")


Processing 2018
Rows raw: 348487
Persons: 182434
       market_income  net_market_income  disposable_income
count   1.824340e+05       1.824340e+05       1.824340e+05
mean    1.481831e+04       1.393093e+04       1.429377e+04
std     4.183999e+04       3.223263e+04       3.214335e+04
min     0.000000e+00       0.000000e+00       0.000000e+00
25%     4.987049e+02       4.891300e+02       1.623910e+03
50%     8.353313e+03       8.192930e+03       8.453030e+03
75%     1.932483e+04       1.848912e+04       1.859015e+04
max     5.415306e+06       3.684098e+06       3.684098e+06
Saved: income_components_2018.csv


# Bloque 5 – Procesamiento ENIGH 2024

In [25]:
# =====================================================
# BLOQUE 5
# PROCESAMIENTO ENIGH 2024
# =====================================================

year = 2024
print(f"\n{'='*30}")
print(f"Processing {year}")
print(f"{'='*30}")

# ---- Load data ----
df = pd.read_csv(
    RAW_DIR / f"ingresos_{year}.csv",
    dtype={
        "folioviv": "string",
        "foliohog": "string",
        "numren": "int64",
        "clave": "string",
        "ing_tri": "float64"
    },
    low_memory=False
)
print(f"Rows raw: {len(df)}")

# ---- Classify (same labor/capital/private, different direct transfers) ----
mask_labor = df["clave"].isin(LABOR_KEYS)   # P020 might be absent but filter works
mask_capital = df["clave"].isin(CAPITAL_KEYS)
mask_private = df["clave"].isin(PRIVATE_TRANSFER_KEYS)
mask_direct = df["clave"].isin(DIRECT_TRANSFERS_2024)

# ---- Aggregate per person ----
net_labor = df.loc[mask_labor].groupby(
    ["folioviv", "foliohog", "numren"]
)["ing_tri"].sum().reset_index(name="net_labor")

mi_capital = df.loc[mask_capital].groupby(
    ["folioviv", "foliohog", "numren"]
)["ing_tri"].sum().reset_index(name="mi_capital")

mi_private = df.loc[mask_private].groupby(
    ["folioviv", "foliohog", "numren"]
)["ing_tri"].sum().reset_index(name="mi_private_transfers")

di_direct = df.loc[mask_direct].groupby(
    ["folioviv", "foliohog", "numren"]
)["ing_tri"].sum().reset_index(name="di_direct_transfers")

# ---- Merge ----
person_base = df[["folioviv", "foliohog", "numren"]].drop_duplicates()
person = person_base.merge(net_labor, on=["folioviv","foliohog","numren"], how="left")
person = person.merge(mi_capital, on=["folioviv","foliohog","numren"], how="left")
person = person.merge(mi_private, on=["folioviv","foliohog","numren"], how="left")
person = person.merge(di_direct, on=["folioviv","foliohog","numren"], how="left")

# Fill NaN with 0
for col in ["net_labor", "mi_capital", "mi_private_transfers", "di_direct_transfers"]:
    person[col] = person[col].fillna(0)

# ---- Gross-up labor ----
brackets = TAX_BRACKETS[year]
person["mi_trabajo"] = gross_up_labor(person["net_labor"], brackets)

# ---- Market income ----
person["market_income"] = (
    person["mi_trabajo"]
    + person["mi_capital"]
    + person["mi_private_transfers"]
)

# ---- ISR ----
person["isr"] = person["market_income"].apply(lambda x: calculate_isr(x, brackets))

# ---- Net market income ----
person["net_market_income"] = person["market_income"] - person["isr"]

# ---- Disposable income ----
person["disposable_income"] = person["net_market_income"] + person["di_direct_transfers"]

# ---- Deflate to 2018 prices ----
deflator = DEFLATORS[year]
monetary_cols = ["mi_trabajo", "mi_capital", "mi_private_transfers",
                 "market_income", "net_market_income",
                 "di_direct_transfers", "disposable_income"]
person[monetary_cols] = person[monetary_cols] * deflator

# ---- Final columns ----
person["year"] = year
df_2024_out = person[["folioviv", "foliohog", "numren", "year", "mi_trabajo", "isr",
                      "mi_capital", "mi_private_transfers",
                      "market_income", "net_market_income",
                      "di_direct_transfers", "disposable_income"]].copy()

# ---- Diagnostics ----
print(f"Persons: {len(df_2024_out)}")
print(df_2024_out[["market_income", "net_market_income", "disposable_income"]].describe())

# ---- Save ----
output_file = PROCESSED_DIR / f"income_components_{year}.csv"
df_2024_out.to_csv(output_file, index=False)
print(f"Saved: {output_file.name}")


Processing 2024
Rows raw: 391563
Persons: 202365
       market_income  net_market_income  disposable_income
count   2.023650e+05       2.023650e+05       2.023650e+05
mean    1.875114e+04       1.749127e+04       1.837830e+04
std     5.866400e+04       4.196450e+04       4.175013e+04
min     0.000000e+00       0.000000e+00       0.000000e+00
25%     1.539466e+03       1.509908e+03       4.395230e+03
50%     1.297780e+04       1.244756e+04       1.318570e+04
75%     2.525350e+04       2.393761e+04       2.417378e+04
max     1.968411e+07       1.291759e+07       1.291759e+07
Saved: income_components_2024.csv


# Bloque 6 – Concatenación 2018 + 2024 y dataset final

In [26]:
# =====================================================
# BLOQUE 6
# MERGE VERTICAL (2018 arriba, 2024 abajo)
# =====================================================

import pandas as pd

# ----- Load both years -----
df_2018 = pd.read_csv(PROCESSED_DIR / "income_components_2018.csv")
df_2024 = pd.read_csv(PROCESSED_DIR / "income_components_2024.csv")

print("2018 shape:", df_2018.shape)
print("2024 shape:", df_2024.shape)

# ----- Concatenate (2018 first, then 2024) -----
df_final = pd.concat([df_2018, df_2024], axis=0, ignore_index=True)

print("\nFinal dataset shape:", df_final.shape)
print("Year distribution:\n", df_final["year"].value_counts().sort_index())

# ----- Diagnostics -----
print("\nMissing values per column:")
print(df_final.isna().sum())

print("\nIncome summary by year:")
print(
    df_final.groupby("year")[
        ["market_income", "net_market_income", "disposable_income"]
    ].describe()
)

# ----- Export final dataset -----
output_final = PROCESSED_DIR / "income_fiscal_incidence_base.csv"
df_final.to_csv(output_final, index=False)
print(f"\nFinal dataset saved: {output_final.name}")

2018 shape: (182434, 12)
2024 shape: (202365, 12)

Final dataset shape: (384799, 12)
Year distribution:
 year
2018    182434
2024    202365
Name: count, dtype: int64

Missing values per column:
folioviv                0
foliohog                0
numren                  0
year                    0
mi_trabajo              0
isr                     0
mi_capital              0
mi_private_transfers    0
market_income           0
net_market_income       0
di_direct_transfers     0
disposable_income       0
dtype: int64

Income summary by year:
     market_income                                                \
             count          mean           std  min          25%   
year                                                               
2018      182434.0  14818.314584  41839.989889  0.0   498.704943   
2024      202365.0  18751.140067  58663.995993  0.0  1539.465848   

                                               net_market_income  \
               50%           75%           max 